# Interpretability and Explanation Stability

## Scientific objective
Generate descriptor/fingerprint contributions and optional neural/graph attributions, while testing stability across seeds, models, similar molecules, and errors.

## Inputs
- Fitted models
- Test predictions
- Selected molecules

## Expected outputs
- `results/explanations/*`
- fingerprint bit/substructure mappings
- explanation-stability summary

## Dependencies
scikit-learn, RDKit; SHAP/Captum optional

## Reproducibility seed
`20260723`. The seed is loaded from `configs/training_config.yaml`; split files and checkpoints are persisted.

## Data and model assumptions
Attributions describe model behavior. They are used for debugging, error analysis, and hypothesis generation—not as proof of biological mechanism or causal toxicophores.

## Validation checks
The executable cells below fail explicitly on missing/inconsistent required artifacts and save machine-readable status records.

## Interpretation of results
Interpret endpoint-level outputs only after checking prevalence, missingness, split integrity, calibration, uncertainty, and applicability-domain coverage. No notebook result is evidence that experimental toxicity testing can be replaced.

## Saved artifacts
Artifacts listed above are written under `data/`, `models/`, `results/`, `figures/`, `tables/`, or `reports/` and are consumed by later notebooks.

## Limitations
Hashed fingerprint bits can collide; post-hoc explanations can be unstable and model-dependent.

## Next notebook
[20_external_validation.ipynb](./20_external_validation.ipynb)

In [1]:
from pathlib import Path
import os, json, warnings
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").exists():
    raise RuntimeError("Run this notebook from the repository root or notebooks directory")
os.chdir(ROOT)

from toxicity_screening.config import load_configs, execution_profile
from toxicity_screening.utils import set_global_seed, require_paths

CONFIGS = load_configs(ROOT)
PROFILE, PROFILE_CONFIG = execution_profile(CONFIGS)
SEED = int(CONFIGS["training_config"]["seed"])
set_global_seed(SEED)
print({"root": str(ROOT), "profile": PROFILE, "seed": SEED})

{'root': 'D:\\Dropbox\\Work\\Learning\\Python\\toxicity_screening_project', 'profile': 'full', 'seed': 20260723}


In [2]:
import joblib
import numpy as np
import pandas as pd

archive = np.load(
    ROOT / "data/processed/morgan_features.npz",
    allow_pickle=False,
)

X = archive["X"]
ids = archive["molecule_id"].astype(str)

if X.shape[0] != len(ids):
    raise ValueError(
        f"Feature/identifier mismatch: "
        f"{X.shape[0]} feature rows versus {len(ids)} molecule IDs"
    )

feature_index = pd.DataFrame(
    {
        "molecule_id": ids,
        "row": np.arange(len(ids), dtype=np.int64),
    }
)

records = pd.read_parquet(
    ROOT / "data/processed/modeling_records.parquet"
).merge(
    feature_index,
    on="molecule_id",
    validate="many_to_one",
)

output_directory = (
    ROOT / "results" / "explanations"
)
output_directory.mkdir(
    parents=True,
    exist_ok=True,
)

outputs = []

for endpoint in records["endpoint"].dropna().unique():
    estimator_path = (
        ROOT
        / "models"
        / "qsar"
        / f"{endpoint}_estimators.joblib"
    )

    if not estimator_path.exists():
        print(
            f"[INTERPRETABILITY] skipped "
            f"endpoint={endpoint} "
            f"reason=estimators_missing",
            flush=True,
        )
        continue

    models = joblib.load(estimator_path)

    model = models.get(
        "logistic_regression"
    )

    if model is None:
        print(
            f"[INTERPRETABILITY] skipped "
            f"endpoint={endpoint} "
            f"reason=logistic_model_missing",
            flush=True,
        )
        continue

    estimator = (
        model.named_steps["model"]
        if hasattr(model, "named_steps")
        else model
    )

    if not hasattr(estimator, "coef_"):
        print(
            f"[INTERPRETABILITY] skipped "
            f"endpoint={endpoint} "
            f"reason=coefficients_unavailable",
            flush=True,
        )
        continue

    coefficients = np.asarray(
        estimator.coef_,
        dtype=float,
    ).ravel()

    number_of_features = min(
        30,
        len(coefficients),
    )

    top_indices = np.argsort(
        np.abs(coefficients)
    )[-number_of_features:][::-1]

    table = pd.DataFrame(
        {
            "endpoint": endpoint,
            "fingerprint_bit": top_indices.astype(int),
            "coefficient": coefficients[top_indices],
            "absolute_coefficient": np.abs(
                coefficients[top_indices]
            ),
            "association_direction": np.where(
                coefficients[top_indices] > 0,
                "positive",
                "negative",
            ),
        }
    )

    endpoint_output = (
        output_directory
        / f"{endpoint}_top_fingerprint_coefficients.csv"
    )

    table.to_csv(
        endpoint_output,
        index=False,
    )

    outputs.append(table)

    print(
        f"[INTERPRETABILITY] completed "
        f"endpoint={endpoint} "
        f"features={len(table)} "
        f"output={endpoint_output}",
        flush=True,
    )

if not outputs:
    raise RuntimeError(
        "No logistic-regression coefficient tables were generated"
    )

all_importance = pd.concat(
    outputs,
    ignore_index=True,
)

summary_path = (
    output_directory
    / "model_feature_summary.csv"
)

all_importance.to_csv(
    summary_path,
    index=False,
)

print(
    f"[INTERPRETABILITY] completed "
    f"endpoints={all_importance['endpoint'].nunique()} "
    f"rows={len(all_importance)} "
    f"output={summary_path}",
    flush=True,
)

display(all_importance.head(20))

[INTERPRETABILITY] completed endpoint=herg_blockade features=30 output=D:\Dropbox\Work\Learning\Python\toxicity_screening_project\results\explanations\herg_blockade_top_fingerprint_coefficients.csv
[INTERPRETABILITY] completed endpoint=ames_mutagenicity features=30 output=D:\Dropbox\Work\Learning\Python\toxicity_screening_project\results\explanations\ames_mutagenicity_top_fingerprint_coefficients.csv
[INTERPRETABILITY] completed endpoint=SR-p53 features=30 output=D:\Dropbox\Work\Learning\Python\toxicity_screening_project\results\explanations\SR-p53_top_fingerprint_coefficients.csv
[INTERPRETABILITY] completed endpoint=SR-ATAD5 features=30 output=D:\Dropbox\Work\Learning\Python\toxicity_screening_project\results\explanations\SR-ATAD5_top_fingerprint_coefficients.csv
[INTERPRETABILITY] completed endpoint=SR-ARE features=30 output=D:\Dropbox\Work\Learning\Python\toxicity_screening_project\results\explanations\SR-ARE_top_fingerprint_coefficients.csv
[INTERPRETABILITY] completed endpoint=SR

,endpoint,fingerprint_bit,coefficient,absolute_coefficient,association_direction
0,herg_blockade,1791,1.213307,1.213307,positive
1,herg_blockade,350,-0.974188,0.974188,negative
2,herg_blockade,1243,0.885893,0.885893,positive
3,herg_blockade,1160,0.881364,0.881364,positive
4,herg_blockade,1199,0.854839,0.854839,positive
5,herg_blockade,1917,-0.820570,0.820570,negative
6,herg_blockade,1756,-0.816984,0.816984,negative
7,herg_blockade,1164,-0.771517,0.771517,negative
8,herg_blockade,389,-0.754333,0.754333,negative
9,herg_blockade,881,0.730749,0.730749,positive


In [3]:
# Stability across endpoints:
# Jaccard overlap among each endpoint's top fingerprint bits.

import pandas as pd

feature_summary_path = (
    ROOT
    / "results"
    / "explanations"
    / "model_feature_summary.csv"
)

# Prefer the in-memory table, but remain robust after a kernel restart.
if "all_importance" in globals():
    feature_summary = all_importance.copy()
elif "all_imp" in globals():
    feature_summary = all_imp.copy()
elif feature_summary_path.exists():
    feature_summary = pd.read_csv(feature_summary_path)
else:
    raise FileNotFoundError(
        "The feature summary is unavailable. Run the preceding "
        f"coefficient-extraction cell first: {feature_summary_path}"
    )

required_columns = {
    "endpoint",
    "fingerprint_bit",
}

missing_columns = required_columns - set(feature_summary.columns)

if missing_columns:
    raise ValueError(
        "Feature summary is missing required columns: "
        f"{sorted(missing_columns)}"
    )

top_sets = {
    endpoint: set(
        group["fingerprint_bit"]
        .dropna()
        .astype(int)
    )
    for endpoint, group in feature_summary.groupby(
        "endpoint",
        sort=False,
    )
}

rows = []

for endpoint_a, bits_a in top_sets.items():
    for endpoint_b, bits_b in top_sets.items():
        intersection_size = len(
            bits_a & bits_b
        )

        union_size = len(
            bits_a | bits_b
        )

        jaccard = (
            intersection_size / union_size
            if union_size
            else float("nan")
        )

        rows.append(
            {
                "endpoint_a": endpoint_a,
                "endpoint_b": endpoint_b,
                "n_bits_a": len(bits_a),
                "n_bits_b": len(bits_b),
                "shared_bits": intersection_size,
                "union_bits": union_size,
                "jaccard_top_bits": jaccard,
            }
        )

stability_summary = pd.DataFrame(rows)

output_path = (
    ROOT
    / "results"
    / "explanations"
    / "explanation_stability_summary.csv"
)

output_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)

stability_summary.to_csv(
    output_path,
    index=False,
)

print(
    f"[STABILITY] completed "
    f"endpoints={len(top_sets)} "
    f"comparisons={len(stability_summary)} "
    f"output={output_path}",
    flush=True,
)

display(
    stability_summary.pivot(
        index="endpoint_a",
        columns="endpoint_b",
        values="jaccard_top_bits",
    )
)

[STABILITY] completed endpoints=6 comparisons=36 output=D:\Dropbox\Work\Learning\Python\toxicity_screening_project\results\explanations\explanation_stability_summary.csv


endpoint_b,SR-ARE,SR-ATAD5,SR-MMP,SR-p53,ames_mutagenicity,herg_blockade
endpoint_a,,,,,,
SR-ARE,1.000000,0.071429,0.132075,0.071429,0.016949,0.016949
SR-ATAD5,0.071429,1.000000,0.111111,0.132075,0.016949,0.016949
SR-MMP,0.132075,0.111111,1.000000,0.111111,0.034483,0.000000
SR-p53,0.071429,0.132075,0.111111,1.000000,0.016949,0.052632
ames_mutagenicity,0.016949,0.016949,0.034483,0.016949,1.000000,0.000000
herg_blockade,0.016949,0.016949,0.000000,0.052632,0.000000,1.000000


### Completion gate
Confirm that the declared artifacts exist before continuing to `20_external_validation.ipynb`.